In [1]:
!pip install pennylane

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 48.8 MB/s eta 0:00:00
  Attempting uninstall: autograd
    Found existing installation: autograd 1.9.1
    Uninstalling autograd-1.9.1:
      Successfully uninstalled autograd-1.9.1


In [2]:
import time
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pennylane as qml
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.datasets import load_wine
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.functional as F

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ==========================================================
# 1. VERİ HAZIRLIĞI (WINE - TÜM VERİLER DAHİL EDİLDİ)
# ==========================================================
print("--- [KOD] 3 SINIFLI KLASİK VE KUANTUM SINIFLANDIRICI (5-FOLD CV) BAŞLATILIYOR ---")

wine = load_wine()
X_raw = wine.data
y_raw = wine.target  # 0, 1, 2 sınıfları mevcut

print(f"Toplam Örneklem: {len(X_raw)} (Sınıf 0: {np.sum(y_raw == 0)}, Sınıf 1: {np.sum(y_raw == 1)}, Sınıf 2: {np.sum(y_raw == 2)})\n")

n_qubits = 4
n_classes = 3

# ==========================================================
# 2. SINIFLANDIRICI MODEL MİMARİLERİ
# ==========================================================
class ClassicalClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        # 4 özellik -> 8 -> 8 -> 3 Sınıf (Sınıflandırma Mimarisi)
        self.net = nn.Sequential(
            nn.Linear(n_qubits, 8),
            nn.ReLU(),
            nn.Linear(8, 8),
            nn.ReLU(),
            nn.Linear(8, n_classes) # Çıktı nöron sayısı 3 (Sınıf sayısı)
        )

    def forward(self, x):
        return self.net(x)

# 2 Katmanlı QNN (Kuantum Sinir Ağı) Devresi
fixed_layers = 2
dev_qnn = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev_qnn, interface="torch", diff_method="backprop")
def qnn_qnode(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(n_qubits))
    qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
    # 3 sınıf için ilk 3 kübitin PauliZ ölçümünü alıyoruz (-1 ile 1 arası değerler döner)
    return [qml.expval(qml.PauliZ(i)) for i in range(n_classes)]

weight_shapes = {"weights": (fixed_layers, n_qubits, 3)}

class QuantumClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.qlayer = qml.qnn.TorchLayer(qnn_qnode, weight_shapes)

    def forward(self, x):
        # Ölçüm sonuçlarını direkt logit (skor) olarak 3 sınıf için döndürüyoruz
        return self.qlayer(x)

# ==========================================================
# 3. 5-FOLD STRATIFIED SINIFLANDIRMA CV MOTORU
# ==========================================================
def run_classifier_cv(model_cls, model_name, epochs=30, lr=0.01):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
    fold_auc, fold_acc, fold_f1 = [], [], []

    print(f"[{model_name}] 5-Fold CV Başlatılıyor ({epochs} Epoch)...")
    start_total_t = time.time()

    # Çok sınıflı problem için kayıp fonksiyonu
    criterion = nn.CrossEntropyLoss()

    for fold, (train_idx, test_idx) in enumerate(skf.split(X_raw, y_raw), 1):
        # 1. Veri Sızıntısını Önleme (PCA ve Scaler sadece Train'e fit edilir)
        X_tr_raw, X_te_raw = X_raw[train_idx], X_raw[test_idx]
        y_tr, y_te = y_raw[train_idx], y_raw[test_idx]

        scaler = StandardScaler()
        pca = PCA(n_components=n_qubits, random_state=RANDOM_SEED)
        minmax = MinMaxScaler(feature_range=(-np.pi, np.pi))

        X_tr_angles = minmax.fit_transform(pca.fit_transform(scaler.fit_transform(X_tr_raw)))
        X_te_angles = minmax.transform(pca.transform(scaler.transform(X_te_raw)))

        # Sınıflandırma eğitim verileri (CrossEntropyLoss için etiketler long tipinde olmalı)
        train_tensor = torch.tensor(X_tr_angles, dtype=torch.float32)
        train_labels = torch.tensor(y_tr, dtype=torch.long)

        test_tensor = torch.tensor(X_te_angles, dtype=torch.float32)
        test_labels = torch.tensor(y_te, dtype=torch.long)

        train_loader = DataLoader(TensorDataset(train_tensor, train_labels), batch_size=32, shuffle=True)

        model = model_cls()
        optimizer = optim.Adam(model.parameters(), lr=lr)

        # Eğitim
        for epoch in range(epochs):
            model.train()
            for bx, by in train_loader:
                optimizer.zero_grad()
                logits = model(bx)
                loss = criterion(logits, by)
                loss.backward()
                optimizer.step()

        # Doğrulama / Test
        model.eval()
        with torch.no_grad():
            test_logits = model(test_tensor)
            # Olasılıklar (ROC-AUC için)
            test_probs = F.softmax(test_logits, dim=1).numpy()
            # En yüksek olasılığa sahip sınıf (Acc ve F1 için)
            test_preds = torch.argmax(test_logits, dim=1).numpy()

        # Çok sınıflı metrik hesaplamaları (OVR: One-vs-Rest, Macro: Tüm sınıfların ortalaması)
        auc = roc_auc_score(y_te, test_probs, multi_class='ovr', average='macro')
        acc = accuracy_score(y_te, test_preds)
        f1 = f1_score(y_te, test_preds, average='macro')

        fold_auc.append(auc)
        fold_acc.append(acc)
        fold_f1.append(f1)

    elapsed_time = time.time() - start_total_t
    print(f"   -> {model_name} Tamamlandı (Süre: {elapsed_time:.1f} sn | AUC: %{np.mean(fold_auc)*100:.2f} ± {np.std(fold_auc)*100:.2f})\n")

    return {
        "auc_mean": np.mean(fold_auc) * 100, "auc_std": np.std(fold_auc) * 100,
        "acc_mean": np.mean(fold_acc) * 100, "acc_std": np.std(fold_acc) * 100,
        "f1_mean": np.mean(fold_f1) * 100, "f1_std": np.std(fold_f1) * 100,
        "time": elapsed_time
    }

# ==========================================================
# 4. MODELLERİN FARKLI EPOCHLARDA YARIŞTIRILMASI
# ==========================================================
print("=== 5-FOLD ÇAPRAZ DOĞRULAMA KOŞUMLARI BAŞLIYOR ===\n")
cv_results = {}

# --- KUANTUM SINIFLANDIRICI (2 KATMAN) ---
cv_results["QNN Sınıflandırıcı (20 Ep)"] = run_classifier_cv(QuantumClassifier, "QNN (20 Ep)", epochs=20, lr=0.02)
cv_results["QNN Sınıflandırıcı (50 Ep)"] = run_classifier_cv(QuantumClassifier, "QNN (50 Ep)", epochs=50, lr=0.02)
cv_results["QNN Sınıflandırıcı (100 Ep)"]= run_classifier_cv(QuantumClassifier, "QNN (100 Ep)", epochs=100, lr=0.02)

# --- KLASİK SINIFLANDIRICI ---
cv_results["Klasik Sınıflandırıcı (20 Ep)"] = run_classifier_cv(ClassicalClassifier, "Klasik NN (20 Ep)", epochs=20, lr=0.005)
cv_results["Klasik Sınıflandırıcı (50 Ep)"] = run_classifier_cv(ClassicalClassifier, "Klasik NN (50 Ep)", epochs=50, lr=0.005)
cv_results["Klasik Sınıflandırıcı (100 Ep)"]= run_classifier_cv(ClassicalClassifier, "Klasik NN (100 Ep)", epochs=100, lr=0.005)

# ==========================================================
# 5. NİHAİ RAPOR TABLOSU
# ==========================================================
print("\n" + "=" * 105)
print("HAKEM RAPORU: 5-FOLD STRATIFIED CV ÇOK SINIFLI SINIFLANDIRMA SONUÇLARI (Mean ± SD)")
print("=" * 105)
print(f"{'Model / Epoch':<30} | {'ROC-AUC (Mean ± SD)':<22} | {'Doğruluk (Mean ± SD)':<22} | {'F1-Skoru (Mean ± SD)'}")
print("-" * 105)
for name, res in cv_results.items():
    auc_str = f"%{res['auc_mean']:.2f} ± {res['auc_std']:.2f}"
    acc_str = f"%{res['acc_mean']:.2f} ± {res['acc_std']:.2f}"
    f1_str = f"%{res['f1_mean']:.2f} ± {res['f1_std']:.2f}"
    print(f"{name:<30} | {auc_str:<22} | {acc_str:<22} | {f1_str}")
print("=" * 105)

--- [KOD] 3 SINIFLI KLASİK VE KUANTUM SINIFLANDIRICI (5-FOLD CV) BAŞLATILIYOR ---
Toplam Örneklem: 178 (Sınıf 0: 59, Sınıf 1: 71, Sınıf 2: 48)

=== 5-FOLD ÇAPRAZ DOĞRULAMA KOŞUMLARI BAŞLIYOR ===

[QNN (20 Ep)] 5-Fold CV Başlatılıyor (20 Epoch)...
   -> QNN (20 Ep) Tamamlandı (Süre: 20.7 sn | AUC: %88.84 ± 4.00)

[QNN (50 Ep)] 5-Fold CV Başlatılıyor (50 Epoch)...
   -> QNN (50 Ep) Tamamlandı (Süre: 26.8 sn | AUC: %89.80 ± 5.31)

[QNN (100 Ep)] 5-Fold CV Başlatılıyor (100 Epoch)...
   -> QNN (100 Ep) Tamamlandı (Süre: 53.3 sn | AUC: %89.78 ± 5.13)

[Klasik NN (20 Ep)] 5-Fold CV Başlatılıyor (20 Epoch)...
   -> Klasik NN (20 Ep) Tamamlandı (Süre: 0.8 sn | AUC: %99.72 ± 0.31)

[Klasik NN (50 Ep)] 5-Fold CV Başlatılıyor (50 Epoch)...
   -> Klasik NN (50 Ep) Tamamlandı (Süre: 2.0 sn | AUC: %99.51 ± 0.47)

[Klasik NN (100 Ep)] 5-Fold CV Başlatılıyor (100 Epoch)...
   -> Klasik NN (100 Ep) Tamamlandı (Süre: 3.8 sn | AUC: %99.78 ± 0.33)


HAKEM RAPORU: 5-FOLD STRATIFIED CV ÇOK SINIFLI SINIFLAND